# v31 — большее разрешение OSNet без нового обучения

**Restart Kernel → Run All** в прежнем research `.venv`. Проверяем, помогают ли признаки R1 на 320/384 px дополнить исходные 256 px. Пять фиксированных вариантов: v25 и два разрешения × две доли новых признаков.

Это новый проход по изображениям, а не перестановка старых оценок. Больше деталей возможно лишь там, где они есть в исходном crop; увеличивать маленькую картинку — не значит восстанавливать потерянные детали. Прирост не гарантирован.

Те же три R1-модели и тот же bbox; веса, batch normalization, порог, кандидаты и MVP не меняются. Выбор — на calibration; validation — контроль и один замороженный выбор. Оба split уже наблюдались: это development-поиск, не независимая оценка скрытого теста.

In [1]:
from pathlib import Path
import os, sys, subprocess
# До загрузки ONNX Runtime: защита от ранее диагностированного native SIGABRT.
if 'onnxruntime' in sys.modules and os.environ.get('ORT_DISABLE_TELEMETRY') != '1':
    raise RuntimeError('ONNX Runtime уже загружен. Выполни Restart Kernel → Run All')
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/multiscale.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import multiscale as experiment
RUN_NAME = 'multiscale_v1'
SOURCE_RUN = 'pool_rerank_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME)
display(experiment.systems())
print('CPU; новые признаки 320/384; без обучения и лимита времени; сохранение каждые 128 изображений')

Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: multiscale_v1


[{'name': 'V25_control', 'size': 256, 'high_weight': 0.0},
 {'name': 'r1_s320_w25', 'size': 320, 'high_weight': 0.25},
 {'name': 'r1_s320_w50', 'size': 320, 'high_weight': 0.5},
 {'name': 'r1_s384_w25', 'size': 384, 'high_weight': 0.25},
 {'name': 'r1_s384_w50', 'size': 384, 'high_weight': 0.5}]

CPU; новые признаки 320/384; без обучения и лимита времени; сохранение каждые 128 изображений


## 1. Проверки до запуска

Проверяем неизменность весов и операций ONNX, геометрию bbox, JPEG/PNG, независимость query, сохранность кандидатов, экспорт, продолжение прерванного извлечения и отказы при повреждении файлов. Сохранённые outputs notebook не мешают повторному Run All. Реальные ошибки тестов останавливают запуск.

In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_multiscale.py', 'tests/test_pool_rerank.py', 'tests/test_dual_role_inference.py'], check=True, env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})

........................................................................ [100%]
72 passed in 23.70s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_multiscale.py', 'tests/test_pool_rerank.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Проверка исходных моделей и завершённого v30

Проверяем контрольные суммы источников и создаём отдельный каталог v31. В ONNX-копиях в памяти меняется только объявленный размер входа; операции, константы и веса остаются прежними. Исходные файлы моделей не перезаписываются. Перед извлечением проверяется воспроизведение R1 на 256 px и совпадение пакетного/одиночного инференса нового размера.

In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Результаты:', context['output'])
print('Signature:', context['signature'])
display(context['manifest']['multiscale_plan'])

Preflight OK: v25 + R1 320/384 at two fixed weights; no training or MVP promotion
Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_31_multiscale/runs/multiscale_v1
Signature: 1f57ad99cfeccfaf0971541cb0ccd097c987879edc6b23846a550334656e1f26


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1',
 'source_signature': '467976b0985cbace3d9915df2b578c755e1abd48bc9cf19815e84e9987813a35',
 'systems': [{'name': 'V25_control', 'size': 256, 'high_weight': 0.0},
  {'name': 'r1_s320_w25', 'size': 320, 'high_weight': 0.25},
  {'name': 'r1_s320_w50', 'size': 320, 'high_weight': 0.5},
  {'name': 'r1_s384_w25', 'size': 384, 'high_weight': 0.25},
  {'name': 'r1_s384_w50', 'size': 384, 'high_weight': 0.5}],
 'source_models': [{'bundle': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_16_review_protocol/runs/review_v1/final/R1_resolution256/seed_20260915/bundle.json',
   'bundle_sha256': '4bb0e340d90a5fab1928885e5fb456e4ab3cac6749238aefe09517a92ee99869',
   'model': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet

## 3. Извлечение → calibration → один выбор → validation

Доли в cosine-fusion: MVP = 50%; R1_256 = 50% × (1 − w); R1_320/384 = 50% × w, где w = 25% или 50%. Граф остаётся 20/3/λ0.50 и использует полную статическую gallery. Отдельный кандидат и порог берутся только из исходной R1_256.

Для calibration извлекаются оба новых размера. Каждый размер используется сразу в двух сравнениях без повторного извлечения. После фиксации победителя для validation извлекается только его размер; если победит v25, нового validation-инференса не будет.

Прогресс: этап, размер, готовые изображения, fresh/cached, номер сравнения, mAP и время. Лимита времени нет. Повторный Restart Kernel → Run All с тем же RUN_NAME продолжит с готовых блоков. Не удаляй receipts и не меняй параметры внутри начатого run.

In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
assert results['status'] == 'complete' and results['protected_unchanged']
assert results['optimizer_updates'] == results['bn_updates'] == 0
assert not results['promoted'] and not results['threshold_fit']
print('Готово. Выбор по calibration:', results['selection']['selected']['name'])
print('Время текущего вызова:', round(results['elapsed_seconds'] / 60, 2), 'мин')


CALIBRATION 1/5 V25_control | elapsed 10.8s
[START] calibration_V25_control | elapsed 0.00 h
calibration V25_control: mAP=0.826410; original R1 candidates unchanged
[DONE] calibration_V25_control | 0.0 min
[START] features_calibration_320 | elapsed 0.00 h
EXTRACT calibration 320px: 128/1202 | fresh 128, cached 0
EXTRACT calibration 320px: 256/1202 | fresh 256, cached 0
EXTRACT calibration 320px: 384/1202 | fresh 384, cached 0
EXTRACT calibration 320px: 512/1202 | fresh 512, cached 0
EXTRACT calibration 320px: 640/1202 | fresh 640, cached 0
EXTRACT calibration 320px: 768/1202 | fresh 768, cached 0
EXTRACT calibration 320px: 896/1202 | fresh 896, cached 0
EXTRACT calibration 320px: 1024/1202 | fresh 1024, cached 0
EXTRACT calibration 320px: 1152/1202 | fresh 1152, cached 0
EXTRACT calibration 320px: 1202/1202 | fresh 1202, cached 0
[DONE] features_calibration_320 | 2.5 min

CALIBRATION 2/5 r1_s320_w25 | elapsed 158.4s
[START] calibration_r1_s320_w25 | elapsed 0.04 h
calibration r1_s320_

## 4. Итог

Для validation сохраняются submission.csv, candidates.csv и реальные embeddings.npy: 2048 измерений у контроля, 3584 у multi-scale. В начале — неизменные исходные 2048; добавлены 1536 новых признаков. Порядок: query, затем gallery. При отказе ranking всё равно содержит десять разных ID.

Дополнительные проходы моделей увеличивают стоимость инференса. Здесь проверяется качество, а не официальный GPU benchmark. Номерная зона специально не обрабатывается; отсутствие остаточного сигнала не доказано. Ограничение junk/top-10 остаётся прежним. Автоматического продвижения в MVP нет.

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
display(results['selected_vs_v25'])
for name, report in results['evaluations'].items():
    print(name, '→', report['export'])
print('Основные файлы:', context['output'] / 'results.json', context['output'] / 'REPORT.md')

# v31 — multi-scale OSNet

Статус: complete
Те же веса R1: 256 + 320/384 px, доля высокого разрешения внутри R1 25/50%.
MVP/R1 остаётся 50/50, граф 20/3/λ0.50. Входной bbox не меняется.

## Calibration: пять вариантов

| Вариант | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| V25_control | 0.826410 | 0.841463 | 0.813483 | 0.704918 |
| r1_s320_w25 | 0.825166 | 0.837398 | 0.813483 | 0.704918 |
| r1_s320_w50 | 0.823482 | 0.833333 | 0.813483 | 0.704918 |
| r1_s384_w25 | 0.824702 | 0.837398 | 0.813483 | 0.704918 |
| r1_s384_w50 | 0.802837 | 0.804878 | 0.813483 | 0.704918 |

## Validation: контроль и один выбор

| Вариант | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| V25_control | 0.828957 | 0.829960 | 0.795455 | 0.709677 |

Выбор: V25_control
ΔmAP против v25: +0.000000; AP лучше/хуже/равно: 0/0/247.

Это адаптивный development-поиск, не независимый тест; прирост не гарантирован.
Веса, bbox, candidate/confidence/threshold, evaluator и рабочий MVP неизменны. Обучения и автоматического продвижения нет.
Изменяется только размер подачи того же изображения; деталей сверх разрешения исходного crop метод не создаёт.
Номерная зона не обрабатывается; отсутствие остаточного сигнала номера не доказано.
Экспорт: реальные float32-векторы, 2048 для контроля / 3584 для multi-scale; top-10 сохраняется при отказах.
Ограничение junk/top-10 сохраняется. Дополнительные encoder passes увеличивают стоимость; это не официальный benchmark.
Источники неизменны: True. Время текущего вызова: 6.0 мин.


{'mAP_delta': 0.0,
 'C_delta': 0.0,
 'improved': 0,
 'worsened': 0,
 'unchanged': 247}

V25_control → /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_31_multiscale/runs/multiscale_v1/tasks/validation_V25_control/export
Основные файлы: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_31_multiscale/runs/multiscale_v1/results.json /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_31_multiscale/runs/multiscale_v1/REPORT.md
